# Movie Recommendation System

This notebook implements a movie recommendation system using the MovieLens dataset. It covers:
1. **Exploratory Data Analysis (EDA)**
2. **Data Preprocessing** (Temporal Split)
3. **Model Implementation** (User-Based CF, Item-Based CF, SVD)
4. **Evaluation** (RMSE, MAE, Precision@k, Recall@k)
5. **Recommendation Generation** (with Cold-Start Strategy)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import train_test_split
from collections import defaultdict

%matplotlib inline
sns.set_style('whitegrid')

## 1. Exploratory Data Analysis (EDA)

Load the dataset and explore its characteristics.

In [ ]:
# Load data
ratings = pd.read_csv('../data/ml-latest-small/ratings.csv')
movies = pd.read_csv('../data/ml-latest-small/movies.csv')

print(f'Ratings shape: {ratings.shape}')
print(f'Movies shape: {movies.shape}')
ratings.head()

In [ ]:
# Visualize rating distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='rating', data=ratings)
plt.title('Distribution of Movie Ratings')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

In [ ]:
# Analyze sparsity
n_users = ratings['userId'].nunique()
n_items = ratings['movieId'].nunique()
n_ratings = ratings.shape[0]

sparsity = 1 - (n_ratings / (n_users * n_items))
print(f'Number of users: {n_users}')
print(f'Number of items: {n_items}')
print(f'Sparsity: {sparsity:.4f}')

## 2. Data Preprocessing

We will perform a temporal train-test split to simulate a realistic scenario.

In [ ]:
# Sort by timestamp
ratings_sorted = ratings.sort_values(by='timestamp')

# Temporal split: Use the last 20% of ratings (by time) as the test set
# Note: A strict temporal split might cut off some users entirely from the train set.
# A common approach for users is to hold out their last N interactions, or split by global time.
# Here we will use a global temporal split for simplicity as per common practice in this context,
# but ensuring we don't just split blindly.

# However, Surprise library expects a specific format. We can manually split or use Surprise's.
# Let's stick to the "chronologically-aware" requirement.
# Strategy: For each user, take their last 20% of ratings for testing.

def temporal_split(df, test_size=0.2):
    df_sorted = df.sort_values(by=['userId', 'timestamp'])
    train_data = []
    test_data = []
    
    for _, group in df_sorted.groupby('userId'):
        n_test = int(len(group) * test_size)
        train_data.append(group.iloc[:-n_test])
        test_data.append(group.iloc[-n_test:])
        
    return pd.concat(train_data), pd.concat(test_data)

train_df, test_df = temporal_split(ratings)
print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')

## 3. Model Implementation

We will now implement User-Based CF, Item-Based CF, and SVD using the `Surprise` library.

In [ ]:
# Load data into Surprise format
reader = Reader(rating_scale=(0.5, 5.0))
train_data = Dataset.load_from_df(train_df[['userId', 'movieId', 'rating']], reader)
full_trainset = train_data.build_full_trainset()

# Testset for evaluation (list of tuples)
test_set = list(test_df[['userId', 'movieId', 'rating']].itertuples(index=False, name=None))

In [ ]:
# 1. User-Based Collaborative Filtering
sim_options_user = {'name': 'cosine', 'user_based': True}
user_based_cf = KNNBasic(sim_options=sim_options_user)
user_based_cf.fit(full_trainset)

In [ ]:
# 2. Item-Based Collaborative Filtering
sim_options_item = {'name': 'cosine', 'user_based': False}
item_based_cf = KNNBasic(sim_options=sim_options_item)
item_based_cf.fit(full_trainset)

In [ ]:
# 3. Matrix Factorization (SVD)
svd = SVD(n_factors=100, random_state=42)
svd.fit(full_trainset)

## 4. Evaluation

We evaluate models using RMSE, MAE, Precision@k, and Recall@k.

In [ ]:
def calculate_metrics(predictions, k=10, threshold=3.5):
    # RMSE and MAE
    rmse = accuracy.rmse(predictions, verbose=False)
    mae = accuracy.mae(predictions, verbose=False)
    
    # Precision and Recall
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()

    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold))
                              for (est, true_r) in user_ratings[:k])

        precisions[uid] = n_rel_and_rec_k / k if k != 0 else 0
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    mean_precision = sum(prec for prec in precisions.values()) / len(precisions)
    mean_recall = sum(rec for rec in recalls.values()) / len(recalls)
    
    return rmse, mae, mean_precision, mean_recall

In [ ]:
models = {'User-Based CF': user_based_cf, 'Item-Based CF': item_based_cf, 'SVD': svd}
results = []

for name, model in models.items():
    print(f'Evaluating {name}...')
    predictions = model.test(test_set)
    rmse, mae, precision, recall = calculate_metrics(predictions, k=10)
    results.append([name, rmse, mae, precision, recall])

results_df = pd.DataFrame(results, columns=['Model', 'RMSE', 'MAE', 'Precision@10', 'Recall@10'])
results_df

## 5. Recommendation Generation & Advanced Features

Generate top-N recommendations and address cold-start.

In [ ]:
def get_recommendations(user_id, model, n=10):
    # Get all movie IDs
    all_movie_ids = set(movies['movieId'].unique())
    
    # Get movies rated by user
    user_rated_movies = set(train_df[train_df['userId'] == user_id]['movieId'].unique())
    
    # Movies to predict
    movies_to_predict = list(all_movie_ids - user_rated_movies)
    
    predictions = []
    for movie_id in movies_to_predict:
        predictions.append((movie_id, model.predict(user_id, movie_id).est))
    
    # Sort by estimated rating
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    top_n = predictions[:n]
    top_n_ids = [x[0] for x in top_n]
    
    return movies[movies['movieId'].isin(top_n_ids)][['title', 'genres']]

# Cold-start strategy: Recommend popular movies (highest average rating with minimum vote count)
def get_cold_start_recommendations(n=10):
    movie_stats = ratings.groupby('movieId').agg({'rating': ['count', 'mean']})
    movie_stats.columns = ['count', 'mean']
    popular_movies = movie_stats[movie_stats['count'] > 50].sort_values(by='mean', ascending=False)
    top_ids = popular_movies.head(n).index
    return movies[movies['movieId'].isin(top_ids)][['title', 'genres']]

# Example Usage
user_id = 1
print(f"Recommendations for User {user_id}:")
display(get_recommendations(user_id, svd))

In [ ]:
# Visualization of SVD Latent Factors (First 2 dimensions)
item_factors = svd.qi
plt.figure(figsize=(10, 8))
plt.scatter(item_factors[:, 0], item_factors[:, 1], alpha=0.5)
plt.title('SVD Item Latent Factors (First 2 Dimensions)')
plt.xlabel('Factor 1')
plt.ylabel('Factor 2')
plt.show()